# LiTFiC — Option Z: run the real pipeline on synthetic tiny data (Kaggle)

Runs the paper's **real** stack (hydra + Lightning + datamodule + LMDB + LoRA + eval) on a **synthetic** BOBSL-format dataset — **no download, no BBC license, no 262 GB**. Proves the *flow*; translations are meaningless (random features).

**Settings:** Accelerator = **GPU T4 ×2** (or ×1), Internet = **On**. Just **Run all**.

Default LLM is the tiny `SmolLM-135M` (ungated, already verified with this repo's decoder) so it runs fast with no HF token. To use the paper-family model, set `LLM='meta-llama/Llama-3.2-3B'` in the config cell (needs an `HF_TOKEN` Kaggle secret + license accept). First run downloads the LLM and the BLEURT eval model.

In [ ]:
# 1) Dependencies (NO flash-attn -- Turing T4 unsupported; we use eager attention).
!pip -q install lightning==2.3.0 hydra-core==1.3.2 hydra-colorlog==1.2.0 \
  transformers==4.45.2 peft==0.12.0 einops lmdb sentencepiece rootutils rich \
  torchmetrics pycocoevalcap bleurt_pytorch nltk \
  pytorch-lightning-bolts==0.3.2.post1 lightning-bolts==0.7.0 lightning-utilities==0.11.2
!apt-get -qq install -y default-jre > /dev/null   # java for pycocoevalcap PTB tokenizer
import nltk
for pkg in ['stopwords','wordnet','averaged_perceptron_tagger','averaged_perceptron_tagger_eng','punkt']:
    try: nltk.download(pkg, quiet=True)
    except Exception as e: print('nltk', pkg, e)
print('deps done')

In [ ]:
# 2) Get the repo. Default: official upstream. If you pushed your fork with the
#    Option-Z scripts, set REPO_URL to it (this notebook inlines the generator, so
#    upstream src/ + configs are all that's needed).
import os
REPO_URL = 'https://github.com/art-jang/Lost-in-Translation-Found-in-Context.git'
REPO_DIR = '/kaggle/working/litfic'
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
print(os.listdir(REPO_DIR))

## CONFIG

In [ ]:
LLM   = 'HuggingFaceTB/SmolLM-135M'   # ungated + verified; or 'meta-llama/Llama-3.2-3B' (needs HF_TOKEN)
OUT   = '/kaggle/working/tiny_bobsl'
DEVICES = '[0]'                        # '[0,1]' to exercise DDP across 2xT4
EPOCHS  = 1

# optional HF token (only needed for gated models like Llama)
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN loaded')
except Exception:
    print('no HF_TOKEN secret (fine for ungated models)')

In [ ]:
# 3) Generate the synthetic BOBSL-format dataset (inline; matches scripts/make_tiny_bobsl.py).
import json, pickle, numpy as np, lmdb
FPS, WIN, STRIDE, FEAT_DIM = 25, 16, 2, 768
BEGIN = WIN//2 - 1
VOCAB = ['hello','world','sign','language','translate','background','person','table','flower',
         'wind','roman','worship','god','man','standing','front','water','city','history','story']
EPISODES = {'ep001':40.0,'ep002':40.0,'ep003':40.0}
os.makedirs(OUT, exist_ok=True)

def n_feat(d): return (int(d*FPS)-BEGIN)//STRIDE + 2
rng = np.random.default_rng(0)
fe = lmdb.open(OUT+'/feats_lmdb', map_size=256*1024**2, subdir=True)
pe = lmdb.open(OUT+'/pl_lmdb', map_size=256*1024**2, subdir=True)
with fe.begin(write=True) as ft, pe.begin(write=True) as pt:
    for ep,dur in EPISODES.items():
        for i in range(n_feat(dur)):
            end=f'{i+1:07d}.np'
            ft.put(f'{ep}/{end}'.encode(), rng.standard_normal(FEAT_DIM).astype(np.float16).tobytes())
            pt.put(f'{ep}_label/{end}'.encode(), rng.integers(0,len(VOCAB),5).astype(np.int64).tobytes())
            pt.put(f'{ep}_prob/{end}'.encode(), rng.random(5).astype(np.float16).tobytes())
fe.close(); pe.close()

rng = np.random.default_rng(1)
ep_n,st,en,su,du,ids=[],[],[],[],[],[]; gid=0
for ep,dur in EPISODES.items():
    t=0.5
    for _ in range(6):
        L=float(rng.uniform(2.0,4.0))
        if t+L>dur-0.5: break
        w=rng.choice(VOCAB,int(rng.integers(4,8)),replace=True)
        ep_n.append(ep); st.append(round(t,2)); en.append(round(t+L,2))
        su.append(' '.join(w)+'.'); du.append(round(L,2)); ids.append(gid); gid+=1; t+=L+0.3
pickle.dump({'episode_name':ep_n,'start':st,'end':en,'subtitle':su,'duration':du,'id':ids}, open(OUT+'/subtitles.pkl','wb'))
pickle.dump({'videos':{'videos':{'T':np.array([int(d*FPS) for d in EPISODES.values()])},'name':[f'{e}.mp4' for e in EPISODES]}}, open(OUT+'/info.pkl','wb'))
pickle.dump({w:i for i,w in enumerate(VOCAB)}, open(OUT+'/vocab.pkl','wb'))
pickle.dump({w:[w] for w in VOCAB}, open(OUT+'/synonyms.pkl','wb'))
pickle.dump({'video':list(EPISODES),'captions':[[' '.join(rng.choice(VOCAB,3)) for _ in range(int(d)+2)] for d in EPISODES.values()]}, open(OUT+'/blip.pkl','wb'))
eps=list(EPISODES); heldout=eps[-1:]
json.dump({'train':eps[:-1],'val':heldout,'public_test':heldout,'test':heldout}, open(OUT+'/subset2episode.json','w'))
json.dump({}, open(OUT+'/train_cap.json','w'))
json.dump({'idx':[0,10_000_000]}, open(OUT+'/val_start_indices.json','w'))
json.dump({'idx':[0]}, open(OUT+'/test_start_indices.json','w'))
print('tiny dataset ready at', OUT)

In [ ]:
# 4) Write configs/paths/tiny.yaml pointing at the synthetic data; get LLM hidden size.
from transformers import AutoConfig
H = AutoConfig.from_pretrained(LLM, token=os.environ.get('HF_TOKEN')).hidden_size
print('LLM hidden size =', H)
paths_yaml = f'''root_dir: ${{oc.env:PROJECT_ROOT}}
data_dir: ${{paths.root_dir}}/data/
log_dir: ${{paths.root_dir}}/logs/
output_dir: ${{hydra:runtime.output_dir}}
work_dir: ${{hydra:runtime.cwd}}
subset2episode: {OUT}/subset2episode.json
vocab_pkl: {OUT}/vocab.pkl
info_pkl: {OUT}/info.pkl
annotations_pkl: {OUT}/pl_lmdb
vid_features_lmdb: {OUT}/feats_lmdb
subtitles_path: {OUT}/subtitles.pkl
aligned_subtitles_path: {OUT}/subtitles.pkl
synonyms_pkl: {OUT}/synonyms.pkl
llm_root: {LLM}
bleurt_path: lucadiliello/BLEURT-20
blip_cap_path: {OUT}/blip.pkl
val_episode_ind_path: {OUT}/val_start_indices.json
test_episode_ind_path: {OUT}/test_start_indices.json
train_cap_path: {OUT}/train_cap.json
spottings_path: null
'''
open(REPO_DIR + '/configs/paths/tiny.yaml','w').write(paths_yaml)
print('wrote configs/paths/tiny.yaml')

In [ ]:
# 5) Run the REAL training pipeline on the synthetic data (a few steps, tiny val).
cmd = f'''cd {REPO_DIR} && PROJECT_ROOT=$(pwd) python src/train.py \
  experiment=vid+pg+prev+bg paths=tiny \
  trainer=gpu trainer.devices={DEVICES} trainer.precision=16-mixed trainer.max_epochs={EPOCHS} \
  trainer.limit_train_batches=6 trainer.limit_val_batches=4 \
  data.batch_size=1 data.num_workers=2 \
  model.net.mm_projector_config.hidden_size={H} \
  model.net.llm_config.decoder_config.attn_implementation=eager'''
print(cmd, '\n')
get_ipython().system(cmd)

In [ ]:
# 6) Inspect generated captions (GT vs prediction). Meaningless text is expected.
import glob, json
caps = sorted(glob.glob(REPO_DIR + '/logs/**/cap.json', recursive=True))
print('cap files:', caps[-3:])
if caps:
    d = json.load(open(caps[-1]))
    for gt, pred in list(zip(d.get('gt', []), d.get('pred', [])))[:5]:
        print('GT  :', gt)
        print('PRED:', pred, '\n')

## What just ran
The real hydra→Lightning→datamodule→LMDB→prompt→LoRA→eval path executed on synthetic BOBSL-format data. You saw the training loss and a generated-caption dump — confirming the **flow**, not paper numbers.

**Next:** for meaningful outputs, swap in a real BOBSL subset (see `docs/litfic-kaggle-runbook.md`, Stage 0+) and set `LLM='meta-llama/Llama-3.2-3B'`.